In [14]:
class CallOperator(BaseModel):
    """Функция возвращает крайний срок подачи документов в МАИ."""

def get_call_operator(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return "вызов оператора"
    return ToolResponse()

In [9]:
# Установка необходимых библиотек
!pip install ragas langchain openai datasets pandas matplotlib

import os
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
    answer_correctness,
    answer_similarity
)
from datasets import Dataset
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from yandex_cloud_ml_sdk import YCloudML
from yandex_cloud_ml_sdk.search_indexes import (
    StaticIndexChunkingStrategy,
    HybridSearchIndexType,
    ReciprocalRankFusionIndexCombinationStrategy,
)
from typing import Optional
from pydantic import BaseModel, Field

# Инициализация Yandex Cloud ML SDK
folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'
sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = sdk.models.completions("yandexgpt", model_version="rc")

# Определение функций и моделей для ассистента
class GetAdmissionDeadlineParams(BaseModel):
    """Функция возвращает крайний срок подачи документов в МАИ."""

def get_admission_deadline(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return "Крайний срок подачи документов в МАИ — 25 июля 2025 года."
    return ToolResponse()

class CallOperator(BaseModel):
    """Функция для вызова оператора."""

def get_call_operator(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return "вызов оператора"
    return ToolResponse()

def printx(string):
    display(Markdown(string))

def create_thread():
    return sdk.threads.create(ttl_days=1, expiration_policy="static")

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

# Класс Agent с модификациями для оценки
class Agent:
    def __init__(self, assistant=None, instruction=None, search_index=None, tools=None):
        self.thread = None
        if assistant:
            self.assistant = assistant
        else:
            if tools:
                self.tools = {x['name']: x['fn'] for x in tools}
                tool_defs = [sdk.tools.function(x['model']) for x in tools]
            else:
                self.tools = {}
                tool_defs = []
            if search_index:
                tool_defs.append(sdk.tools.search_index(search_index))
            self.assistant = create_assistant(model, tool_defs)
        if instruction:
            self.assistant.update(instruction=instruction)

    def get_thread(self, thread=None):
        if thread:
            return thread
        if self.thread is None:
            self.thread = create_thread()
        return self.thread

    def __call__(self, message, thread=None):
        thread = self.get_thread(thread)
        thread.write(message)
        run = self.assistant.run(thread)
        res = run.wait()
        if res.tool_calls:
            result = []
            for f in res.tool_calls:
                print(f" + Вызов функции: {f.function.name}, args={f.function.arguments}")
                fn = self.tools[f.function.name]
                obj = fn(**f.function.arguments)
                x = obj.process(thread)
                result.append({"name": f.function.name, "content": x})
            run.submit_tool_results(result)
            res = run.wait()
        return res.text

    def restart(self):
        if self.thread:
            self.thread.delete()
            self.thread = sdk.threads.create(name="Test", ttl_days=1, expiration_policy="static")

    def done(self, delete_assistant=False):
        if self.thread:
            self.thread.delete()
        if delete_assistant:
            self.assistant.delete()

# Создание тестового набора данных для оценки
test_data = {
    "question": [
        "Когда крайний срок подачи документов в МАИ?",
        "Как мне узнать больше о поступлении?",
        "Какие документы нужны для поступления?",
        "Какой проходной балл на бюджет?",
        "Есть ли у вас общежитие для иногородних?"
    ],
    "context": [
        "Крайний срок подачи документов в МАИ — 25 июля 2025 года.",
        "Для получения подробной информации о поступлении можно обратиться к оператору.",
        "Для поступления необходимы: паспорт, аттестат, результаты ЕГЭ, медицинская справка.",
        "Проходной балл на бюджет зависит от факультета и года, в среднем от 240 до 280 баллов.",
        "МАИ предоставляет общежитие для иногородних студентов, количество мест ограничено."
    ],
    "answer": [
        "Крайний срок подачи документов в МАИ — 25 июля 2025 года.",
        "Для получения подробной информации о поступлении вы можете вызвать оператора.",
        "Для поступления необходимы: паспорт, аттестат, результаты ЕГЭ, медицинская справка.",
        "Проходной балл на бюджет зависит от факультета и года, в среднем от 240 до 280 баллов.",
        "МАИ предоставляет общежитие для иногородних студентов, количество мест ограничено."
    ]
}

dataset = Dataset.from_dict(test_data)

# Инициализация ассистента
instruction = """
Ты — сотрудник приёмной комиссии МАИ. Отвечай только на вопросы по поступлению.
Если уместно — используй функцию для вызова оператора и вызови только функцию CallOperator.
Если уместно — используй функцию для определения крайнего срока подачи документов.
Если пользователь не может получить ответ или , то напомни ему что он может вызвать оператора
"""

agent = Agent(
    instruction=instruction,
    tools=[{
        "name": "GetAdmissionDeadlineParams",
        "model": GetAdmissionDeadlineParams,
        "fn": get_admission_deadline
    }, {
        "name": "CallOperator",
        "model": CallOperator,
        "fn": get_call_operator
    }]
)

# Функция для получения ответов от ассистента
def get_agent_responses(questions):
    responses = []
    for question in questions:
        response = agent(question)
        responses.append(response)
    return responses

# Получаем ответы от ассистента
agent_responses = get_agent_responses(test_data["question"])

# Добавляем ответы в датасет
evaluation_data = {
    "question": test_data["question"],
    "context": test_data["context"],
    "answer": agent_responses,
    "ground_truth": test_data["answer"]
}

evaluation_dataset = Dataset.from_dict(evaluation_data)

# Определение метрик для оценки
metrics = [
    faithfulness,  # Оценка точности и достоверности ответов
    answer_relevancy,  # Оценка релевантности ответов
    context_relevancy,  # Оценка релевантности контекста
    answer_correctness,  # Оценка корректности ответов
    answer_similarity  # Оценка схожести с эталонными ответами
]

# Выполнение оценки
result = evaluate(
    evaluation_dataset,
    metrics=metrics,
    llm="gpt-3.5-turbo",
    embeddings="text-embedding-ada-002"
)

# Анализ и визуализация результатов
results_df = pd.DataFrame(result)

print("Средние значения метрик:")
print(results_df.mean())

plt.figure(figsize=(12, 6))
results_df.mean().plot(kind='bar')
plt.title('Средние значения метрик оценки ассистента')
plt.ylabel('Значение')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Вывод детальных результатов
print("\nДетальные результаты по каждому вопросу:")
for idx, row in results_df.iterrows():
    print(f"\nВопрос {idx + 1}: {test_data['question'][idx]}")
    print(f"Ответ ассистента: {agent_responses[idx]}")
    print(f"Оценки: {row.to_dict()}")

In [20]:
instruction = """
Ты — сотрудник приёмной комиссии МАИ. Отвечай только на вопросы по поступлению.
Также храни историю сообщений с пользователем, чтобы если пользователь начнет отвечать на заданные твои вопросы, ты понимал контекст их
Если уместно — используй функцию для вызова оператора и вызови только функцию CallOperator.
Если уместно — используй функцию для определения крайнего срока подачи документов.
Если пользователь не может получить ответ на свой вопрос, то напомни ему что он может вызвать оператора
"""
agent = Agent(
    instruction=instruction,
    search_index=index,
    tools=[{
        "name": "GetAdmissionDeadlineParams",
        "model": GetAdmissionDeadlineParams,
        "fn": get_admission_deadline
    }, {
        "name": "CallOperator",
        "model": CallOperator,
        "fn": get_call_operator
    }]
)

response = agent("я не получил ответ на свой вопрос")
printx(response)

Вы можете задать свой вопрос ещё раз, и я постараюсь ответить на него. Если же ваш вопрос требует более детального обсуждения, вы можете воспользоваться функцией прямого звонка в call-центр приёмной комиссии. График работы: Пн-Пт: 10:00–17:00; Сб: 10:00–14:00. Или же я могу помочь вам с вызовом оператора прямо сейчас.

In [22]:
response = agent("я не получил ответ на свой вопрос")
printx(response)
response = agent("можно оператора?")
printx(response)

Вы можете задать свой вопрос ещё раз, и я постараюсь ответить на него. Если же ваш вопрос требует более детального обсуждения, вы можете воспользоваться функцией прямого звонка в call-центр приёмной комиссии. График работы: Пн-Пт: 10:00–17:00; Сб: 10:00–14:00. Или же я могу помочь вам с вызовом оператора прямо сейчас.

 + Вызов функции: CallOperator, args={}


Оператор скоро свяжется с вами для ответа на ваши вопросы.

In [ ]:
"import os\n",
    "from ragas import evaluate\n",
    "from ragas.metrics import (\n",
    "    faithfulness,\n",
    "    answer_relevancy,\n",
    "    context_relevancy,\n",
    "    context_recall,\n",
    "    answer_correctness,\n",
    "    answer_similarity\n",
    ")\n",
    "from datasets import Dataset\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from IPython.display import Markdown, display\n",
    "from yandex_cloud_ml_sdk import YCloudML\n",
    "from yandex_cloud_ml_sdk.search_indexes import (\n",
    "    StaticIndexChunkingStrategy,\n",
    "    HybridSearchIndexType,\n",
    "    ReciprocalRankFusionIndexCombinationStrategy,\n",
    ")\n",
    "from typing import Optional\n",
    "from pydantic import BaseModel, Field"